# LDS_101：在 LDS 里放数据结构和 tensor

LDS（Local Data Share）就是 HIP 里的 `__shared__`：同一个 workgroup 的 wave 共享。一个 workgroup 最多用 64 KB（gfx942）或 160 KB（gfx950，本机 `rocminfo` 的 GROUP segment）。[v1/v2](../kernels/gemm/v1.py) 都是先声明一个 `@fx.struct`，再 `fx.SharedAllocator().allocate(...)` 拿到 LDS，最后在上面建 tensor 做 swizzle 读写。本实验把这条路拆开：

1. 怎样像 C++ 一样定义结构体，size/align/offset 怎么算；
2. `SharedAllocator` 的 `static=True/False` 在 LDS 里怎样摆放这些字段；
3. LDS tensor 的 `.load()/.store()` 变成哪些 `ds_*` 指令；
4. bank conflict 怎么来的；
5. 参考 [pyhip/docs/swizzle.ipynb](https://github.com/tingqli/pyhip/blob/main/docs/swizzle.ipynb)，用 FlyDSL 的 `Swizzle` 消除 bank conflict，几种写法的 ISA 有什么不同；
6. 在本机 GPU 上用 rocprofv3 计数器验证；
7. 不经过寄存器、从 HBM 直接搬到 LDS 的几种写法（LDS DMA）及其同步/异步等待。

**关于架构**：本仓库的目标是 gfx942（MI300），本机 GPU 是 gfx950（MI350）。

| 内容 | gfx942 | gfx950 |
|---|---|---|
| ISA（汇编、指令数、VGPR） | `ARCH=gfx942 COMPILE_ONLY=1` 只编译 | `ARCH=gfx950 COMPILE_ONLY=1` 只编译 |
| 数值正确性、rocprofv3 计数器 | 没有硬件，未运行 | 本机实测 |
| bank conflict 模型（第 4 节） | 按 CK 文档的 gfx942 规则推算 | 没有建模，只看计数器 |

每处 ISA 都把两种架构并排列出。gfx942 的汇编没有在 gfx942 上跑过，它的正确性是从同一份源码在 gfx950 上的结果推断的。

## 0. 准备

和 [tensor_101](tensor_101.ipynb) 一样的几个工具，另外：

- `isa(launcher, ..., arch=...)` 编译指定架构；`isa_both(build, ...)` 分别编译 gfx942 / gfx950；`side(...)` 把两份汇编并排打印；
- `ds(text)`：只取 `ds_*` 指令（LDS 读写）；
- `meta(text, key)`：从 kernel metadata 读一个数，如 `group_segment_fixed_size`（静态 LDS 字节数）、`vgpr_count`；
- `host()`：在 host 上开一个 MLIR context，用来直接计算 layout（见 [layout_algebra](layout_algebra.ipynb)）。

In [1]:
import os
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"   # 不用磁盘缓存，保证每次都重新 trace

import contextlib, csv, glob, inspect, io, itertools, re, shutil, subprocess, sys, tempfile
from collections import Counter
from importlib.metadata import version

import torch
import flydsl.compiler as flyc
import flydsl.expr as fx
from flydsl.expr import gpu
from flydsl.compiler.protocol import dsl_size_of, dsl_align_of
from flydsl._mlir import ir

ARCHS = ("gfx942", "gfx950")      # ISA 两种都编译（只编译不运行）
LOCAL_ARCH = torch.cuda.get_device_properties(0).gcnArchName.split(":")[0]   # 数值和计数器在这个 GPU 上跑
stream = fx.Stream(torch.cuda.current_stream().cuda_stream)
print("flydsl", version("flydsl"), "| torch", torch.__version__, "| 本机 GPU", LOCAL_ARCH)


def make_launcher(kernel_fn):
    kernel = flyc.kernel(known_block_size=(64, 1, 1))(kernel_fn)

    @flyc.jit
    def launch(A: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        kernel(A, O).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)

    return launch


def run(launcher, A, O):
    with contextlib.redirect_stdout(io.StringIO()):   # trace 时的 print 只在 isa() 里显示一次
        launcher(A, O, stream)
    torch.cuda.synchronize()


def isa(launcher, *args, arch="gfx942", quiet=False):
    dump_dir = tempfile.mkdtemp(prefix="lds101_")
    env = dict(ARCH=arch, COMPILE_ONLY="1", FLYDSL_DUMP_IR="1", FLYDSL_DUMP_DIR=dump_dir)
    old = {k: os.environ.get(k) for k in env}
    os.environ.update(env)
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            flyc.compile(launcher, *args)
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k)
            else:
                os.environ[k] = v
    for line in out.getvalue().splitlines():
        if not quiet and not line.startswith("[flydsl.compile]"):
            print(line)
    path, = glob.glob(f"{dump_dir}/**/*final_isa.s", recursive=True)
    return open(path).read()


def isa_both(build, *args):
    """build() 每次新建 launcher，分别编译 gfx942 / gfx950；trace 时的 print 只显示一次"""
    return {arch: isa(build(), *args, arch=arch, quiet=i > 0) for i, arch in enumerate(ARCHS)}


def side(columns, width=44):
    """把几列文本并排打印，columns = {标题: 行列表}"""
    print(" | ".join(f"{name:{width}}" for name in columns).rstrip())
    for row in itertools.zip_longest(*columns.values(), fillvalue=""):
        print(" | ".join(f"{c:{width}}" for c in row).rstrip())


def body(text):
    lines, inside = [], False
    for raw in text.splitlines():
        if re.match(r"^\w+:", raw):
            inside = True
            continue
        line = raw.split(";")[0].strip()
        if inside and line and (not line.startswith(".") or line.startswith(".LBB")):
            lines.append(line)
        if line.startswith("s_endpgm"):
            break
    return lines


def ds(text):
    return [l for l in body(text) if l.startswith("ds_")]


def meta(text, key):
    return int(re.search(rf"\.{key}:\s+(\d+)", text).group(1))


@contextlib.contextmanager
def host():
    with ir.Context() as ctx, ir.Location.unknown():
        ctx.load_all_available_dialects()
        with ir.InsertionPoint(ir.Module.create().body):
            yield

flydsl 0.3.4.1 | torch 2.9.1+rocm7.2.0.git7e1940d4 | 本机 GPU gfx950


## 1. 像 C++ 一样定义结构体

下面这段 C++：

```cpp
struct Header { int32_t flag; int16_t cnt; };   // size 8, align 4
union Scratch { float c[64]; bf16 d[128]; };     // size 256, align 4
struct Smem {
    Header  hdr;              // offset 0
    alignas(16) bf16 a[64];   // offset 16
    int32_t x;                // offset 144
    Scratch u;                // offset 148
};                            // 404 补齐到 16 -> size 416, align 16
```

在 FlyDSL 里写成：

In [2]:
@fx.struct
class Header:
    flag: fx.Int32
    cnt: fx.Int16


@fx.union
class Scratch:
    c: fx.Array[fx.Float32, 64]
    d: fx.Array[fx.BFloat16, 128]


@fx.struct
class Smem:
    hdr: Header
    a: fx.Array[fx.BFloat16, 64, 16]   # 第三个参数是对齐字节数，相当于 alignas(16)
    x: fx.Int32
    u: Scratch


for T in [Header, Scratch, Smem]:
    print(f"{T.__name__:8} size={dsl_size_of(T):4} align={dsl_align_of(T)}")

Header   size=   8 align=4
Scratch  size= 256 align=4
Smem     size= 416 align=16


- 字段用类型注解声明，顺序就是内存顺序；`fx.Array[T, N]` 是定长数组，`fx.Array[T, N, align]` 额外指定对齐；单个标量要加对齐用 `fx.Align[T, n]`。
- struct 可以嵌套，`@fx.union` 的各字段共用同一块内存（size 取最大）。
- size/align 的规则和 C 一样：每个字段放到自己对齐的位置，总大小补齐到最大对齐。这里只是类型，还没有分配任何内存。

## 2. 用 `SharedAllocator` 在 LDS 里分配

`fx.SharedAllocator().allocate(Smem)` 返回 `Storage[Smem]`，是一个指向 LDS 的带类型句柄：

- `st.hdr.flag` 得到 `Storage[Int32]`，`.poke(v)` 写，`.peek()` 读；
- `st.a.peek()` 得到 `Array`：只是一个 LDS 指针，不读数据，可以 `a[i]` 读写单个元素，或 `a.view(layout)` 建 tensor；
- v1/v2 写的 `allocate(...).peek()` 会对每个标量字段都生成一次读，不用的会被编译器删掉。

`SharedAllocator(static=...)` 有两种摆放方式，下面同一个 kernel 各编译一次。每个 lane 写 `a[tid]`、`u.c[tid]`，再读回别的 lane 写的值，在 GPU 上检查结果：

In [3]:
def use_smem(static):
    def kernel_fn(A: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        st = fx.SharedAllocator(static=static).allocate(Smem)
        st.hdr.flag.poke(fx.Int32(1))
        st.hdr.cnt.poke(fx.Int16(2))
        st.x.poke(fx.Int32(3))
        a = st.a.peek()
        c = st.u.c.peek()
        a[tid] = A[tid]
        c[tid] = fx.Float32(fx.Int32(tid))
        gpu.barrier()
        h = st.hdr.peek()
        O[tid] = (fx.Float32(h.flag) + fx.Float32(h.cnt) + fx.Float32(st.x.peek())
                  + fx.Float32(a[63 - tid]) + c[(tid + 1) % 64])
        if fx.const_expr(static):
            print("st.a:", st.a, "\na.ptr:", a.ptr.type, "\na.view(64:1):", a.view(fx.make_layout(64, 1)).type)

    return make_launcher(kernel_fn)


A = torch.randn(64, dtype=torch.bfloat16, device="cuda")
O = torch.zeros(64, dtype=torch.float32, device="cuda")
lane = torch.arange(64, device="cuda")
expect = 6 + A.float().flip(0) + (lane + 1) % 64

for static in [False, True]:
    print(f"==== static={static}")
    texts = isa_both(lambda: use_smem(static), A, O, stream)
    side({arch: ds(t) + [f"group_segment_fixed_size = {meta(t, 'group_segment_fixed_size')}"]
          for arch, t in texts.items()})
    O.zero_()
    run(use_smem(static), A, O)
    print(f"本机 {LOCAL_ARCH} GPU 结果正确:", torch.equal(O, expect))

==== static=False
gfx942                                       | gfx950
ds_write_b32 v3, v2                          | ds_write_b32 v3, v2
ds_write_b16 v3, v4 offset:4                 | ds_write_b16 v3, v4 offset:4
ds_write_b32 v3, v5 offset:144               | ds_write_b32 v3, v5 offset:144
ds_write_b32 v1, v2 offset:148               | ds_write_b32 v1, v2 offset:148
ds_write_b16 v6, v7 offset:16                | ds_write_b16 v6, v7 offset:16
ds_read_b32 v1, v3                           | ds_read_b32 v1, v3
ds_read_i16 v2, v3 offset:4                  | ds_read_i16 v2, v3 offset:4
ds_read_b32 v3, v3 offset:144                | ds_read_b32 v3, v3 offset:144
ds_read_u16 v4, v4 offset:16                 | ds_read_u16 v4, v4 offset:16
ds_read_b32 v5, v5 offset:148                | ds_read_b32 v5, v5 offset:148
group_segment_fixed_size = 0                 | group_segment_fixed_size = 0
本机 gfx950 GPU 结果正确: True
==== static=True
st.a: Storage[Array[BFloat16, 64, 16]](Pointer<i8, shared, alig

**`static=False`**（相当于 `extern __shared__`）：所有字段在一块 dynamic LDS 里，offset 严格按 C 规则：

| 字段 | ISA 里的 offset | C++ offset |
|---|---|---|
| `hdr.flag` | `ds_write_b32 ...`（0） | 0 |
| `hdr.cnt` | `offset:4` | 4 |
| `a` | `offset:16` | 16 |
| `x` | `offset:144` | 144 |
| `u.c` | `offset:148` | 148 |

LDS 大小在 launch 时传入，所以 metadata 里的 `group_segment_fixed_size` 是 0。

**`static=True`**（默认，相当于每个字段各写一个 `__shared__`）：每个叶子字段（标量、数组、union）各自成为一个 LDS 全局变量（LLVM IR 里的 `@__shared_alloc_N`），地址由 LLVM 分配。struct 只起分组作用，字段顺序和 padding 都不保留：这里 `a` 在 0、`u` 在 128、`flag`/`x` 挨着放在 384（两个 `poke` 还合并成了一条 `ds_write_b64`）、`cnt` 在 392。metadata 里的 394 B 就是实际用量，没有用到的字段不占空间。

两种方式下，地址都是 "VGPR + 16 位立即数 offset"，常量部分都进了 `offset:`。gfx942 和 gfx950 生成的指令完全相同。只要不依赖字段之间的相对位置，用默认的 `static=True` 即可，v1/v2 就是这样用的。

## 3. LDS tensor 的 `.load()/.store()` 生成什么指令

`a.view(layout)` 得到 `!fly.memref<bf16, shared, ...>`，和 global tensor 用法完全相同，只是地址空间是 `shared`。和 [tensor_101](tensor_101.ipynb) 第 5 节一样，指令宽度由子 tensor 的 layout 决定。

下面每个 lane 先把 A 的一行（64 个 bf16，连续）写进 LDS，再从 LDS 读 `n` 个元素、间隔 `stride`：

In [4]:
@fx.struct
class Tile:
    a: fx.Array[fx.BFloat16, 64 * 64, 16]


def lds_read(n, stride):
    def kernel_fn(A: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        rows = fx.make_ordered_layout((64, 64), (1, 0))     # [64 行][64 列] 行主序
        T = fx.SharedAllocator().allocate(Tile).peek().a.view(rows)
        G = fx.make_view(fx.get_iter(A), rows)
        Ov = fx.make_view(fx.get_iter(O), rows)
        T[tid, None].store(G[tid, None].load())
        gpu.barrier()
        X = fx.make_view(fx.get_iter(T[(tid + 1) % 8, None]), fx.make_layout(n, stride))
        if fx.const_expr(n == 8 and stride == 1):
            print("T[tid, None]:", T[tid, None].type)
        fx.make_view(fx.get_iter(Ov[tid, None]), fx.make_layout(n, 1)).store(X.load())

    return make_launcher(kernel_fn)


A = torch.randn(64, 64, dtype=torch.bfloat16, device="cuda")
O = torch.zeros_like(A)
flat = A.flatten()
for n, stride in [(8, 1), (4, 1), (2, 1), (1, 1), (8, 64)]:
    texts = isa_both(lambda: lds_read(n, stride), A, O, stream)
    ops = {arch: dict(Counter(l.split()[0] for l in ds(t) if l.startswith("ds_read"))) for arch, t in texts.items()}
    O.zero_()
    run(lds_read(n, stride), A, O)
    idx = ((lane[:, None] + 1) % 8) * 64 + torch.arange(n, device="cuda") * stride
    print(f"读 {n:2}:{stride:<2} -> {ops}   GPU 正确: {torch.equal(O[:, :n], flat[idx])}")

side({arch: ds(t) for arch, t in isa_both(lambda: lds_read(8, 64), A, O, stream).items()})

T[tid, None]: !fly.memref<bf16, shared, 64:1, align<16>>
读  8:1  -> {'gfx942': {'ds_read_b128': 1}, 'gfx950': {'ds_read_b128': 1}}   GPU 正确: True
读  4:1  -> {'gfx942': {'ds_read_b64': 1}, 'gfx950': {'ds_read_b64': 1}}   GPU 正确: True
读  2:1  -> {'gfx942': {'ds_read_b32': 1}, 'gfx950': {'ds_read_b32': 1}}   GPU 正确: True
读  1:1  -> {'gfx942': {'ds_read_u16': 1}, 'gfx950': {'ds_read_u16': 1}}   GPU 正确: True
读  8:64 -> {'gfx942': {'ds_read_u16': 8}, 'gfx950': {'ds_read_u16': 8}}   GPU 正确: True
gfx942                                       | gfx950
ds_write_b128 v4, v[6:9] offset:96           | ds_write_b128 v4, v[6:9] offset:96
ds_write_b128 v4, v[10:13] offset:112        | ds_write_b128 v4, v[10:13] offset:112
ds_write_b128 v4, v[14:17] offset:64         | ds_write_b128 v4, v[14:17] offset:64
ds_write_b128 v4, v[18:21] offset:80         | ds_write_b128 v4, v[18:21] offset:80
ds_write_b128 v4, v[22:25] offset:32         | ds_write_b128 v4, v[22:25] offset:32
ds_write_b128 v4, v[26:29] offset

- 两种架构结果相同。写一行 64 个连续 bf16（128 B）是 8 条 `ds_write_b128`，地址共用一个 VGPR，差别都在 `offset:16, 32, ...` 里。
- 连续读 8/4/2/1 个 bf16 分别是 `ds_read_b128` / `ds_read_b64` / `ds_read_b32` / `ds_read_u16`。
- 间隔 64 个元素（读一列）的 `8:64` 只能拆成 8 条 `ds_read_u16`，行距 128 B 都进了立即数 offset。
- LDS 读写的等待用 `s_waitcnt lgkmcnt(...)`。这里 workgroup 只有 1 个 wave，`gpu.barrier()` 只剩 `s_waitcnt lgkmcnt(0)`，没有 `s_barrier`。

## 4. bank conflict

gfx942 的 LDS 有 32 个 bank，每个 4 B，地址 `addr` 落在 bank `(addr / 4) % 32`。按 [CK 文档](https://rocm.docs.amd.com/projects/composable_kernel/en/latest/conceptual/ck_tile/hardware/lds_bank_conflicts.html)（pyhip swizzle.ipynb 也引用了它），`ds_read_b128`/`ds_write_b128` 每个 lane 访问 16 B，占连续 4 个 bank，可以看成 8 个 "16 B bank"；一个 wave 的 64 个 lane 分 8 组，每组 8 个 lane 同一拍访问：

- `ds_write_b128`：lane 0-7、8-15、…、56-63；
- `ds_read_b128`：lane 0-3+20-23、4-7+16-19、8-11+28-31、12-15+24-27、32-35+52-55、36-39+48-51、40-43+60-63、44-47+56-59。

同一组里如果有几个 lane 访问同一个 16 B bank 的不同地址，就要分几拍完成，这就是 bank conflict。下面的模型只适用于 gfx942；gfx950 的 LDS 容量和带宽都变了，这里不建模，只在第 6 节用计数器实测。

以 v1 的 LDS tile 为例：64 行 × 64 个 bf16，一行 128 B 正好是 8 个 16 B chunk、铺满 32 个 bank。

- 写：从 global 读时为了合并访存，8 个相邻 lane 写同一行的 8 个 chunk；
- 读：MFMA 16x16x16 的 A 操作数，lane `l` 读第 `l % 16` 行的第 `l // 16` 个 chunk。

用 Python 按上面的规则算每组需要几拍：

In [5]:
WRITE_GROUPS = [list(range(g, g + 8)) for g in range(0, 64, 8)]
READ_GROUPS = [list(range(a, a + 4)) + list(range(b, b + 4))
               for a, b in [(0, 20), (4, 16), (8, 28), (12, 24), (32, 52), (36, 48), (40, 60), (44, 56)]]


def cycles(addrs, groups):
    """每组 8 个 lane 需要的拍数：落在同一个 16 B bank 的不同地址个数的最大值"""
    out = []
    for g in groups:
        banks = Counter((addr // 16) % 8 for addr in {addrs[l] for l in g})
        out.append(max(banks.values()))
    return out


def phys(c, r, swizzle):
    """第 r 行、第 c 个 chunk 的字节地址"""
    return r * 128 + 16 * ((c ^ (r % 8)) if swizzle else c)


def write_addrs(p, swizzle):   # 第 p 次写：lane t 写第 8p + t//8 行的第 t%8 个 chunk
    return [phys(t % 8, 8 * p + t // 8, swizzle) for t in range(64)]


def read_addrs(i, swizzle):    # 第 i 次读：lane l 读第 16i + l%16 行的第 l//16 个 chunk
    return [phys(l // 16, 16 * i + l % 16, swizzle) for l in range(64)]


print("ds_write_b128 每组拍数:", cycles(write_addrs(0, False), WRITE_GROUPS))
print("ds_read_b128  每组拍数:", cycles(read_addrs(0, False), READ_GROUPS))
print("read 第 0 组 (lane 0-3, 20-23) 的 16B bank:", [(read_addrs(0, False)[l] // 16) % 8 for l in READ_GROUPS[0]])

ds_write_b128 每组拍数: [1, 1, 1, 1, 1, 1, 1, 1]
ds_read_b128  每组拍数: [4, 4, 4, 4, 4, 4, 4, 4]
read 第 0 组 (lane 0-3, 20-23) 的 16B bank: [0, 0, 0, 0, 1, 1, 1, 1]


- 写没有冲突：一组 8 个 lane 正好写一行的 8 个 chunk。
- 读是 4-way conflict：一组里 lane 0-3 读第 0-3 行的 chunk 0，lane 20-23 读第 4-7 行的 chunk 1。行距 128 B 是 bank 周期的整数倍，不同行的同一个 chunk 落在同一个 bank 上，每组要 4 拍。

## 5. swizzle

### 5.1 思路

pyhip 的结论：以 16 B 为单位，把第 `r` 行的第 `c` 个 chunk 放到物理位置 `c ^ (r % 8)`。每一行内只是重排 chunk，所以写一行时 8 个 lane 仍落在 8 个不同 bank；读同一个 chunk 的 8 行时，物理 chunk 变成 8 个不同的值。

### 5.2 FlyDSL 的 `Swizzle`

CuTe 的 `Swizzle<B, M, S>` 是作用在 offset 上的函数：把 offset 第 `[M+S, M+S+B)` 位异或到第 `[M, M+B)` 位，

$$\text{swizzle}(o) = o \oplus \big((o \gg S)\ \&\ ((2^B - 1) \ll M)\big)$$

FlyDSL 写作 `fx.SwizzleType.get(B, M, S)`，再用 `fx.make_composed_layout(fx.static(swz), layout)` 接在一个普通 layout 后面：先按 layout 算出 offset，再做 swizzle。

把 tile 写成 `(8, 8, 64):(1, 8, 64)`，即 (chunk 内元素, chunk, 行)，元素 offset = $e + 8c + 64r$：

- $M = 3$：低 3 位是 chunk 内的 8 个 bf16（16 B），保持不变；
- $S = 3$，$B = 3$：把第 6-8 位（$r \bmod 8$）异或到第 3-5 位（$c$）。

所以 `Swizzle(3, 3, 3)` 正好是 `c ^ (r % 8)`。在 host 上验证，并画出 pyhip 那张表：第 m 行第 k 个 chunk 由 lane `m + 16k` 读，看它被放到了哪个物理 chunk。

In [6]:
with host():
    swz = fx.static(fx.SwizzleType.get(3, 3, 3))
    L = fx.make_composed_layout(swz, fx.make_ordered_layout((8, 8, 64), 0))
    print("L =", L.type)
    same = all(fx.crd2idx((e, c, r), L).to_py_value() == e + 8 * (c ^ (r % 8)) + 64 * r
               for e in range(8) for c in range(8) for r in range(64))
    print("与 e + 8*(c ^ (r%8)) + 64*r 相同:", same)

    for m in range(8):
        line = ["______"] * 8
        for k in range(4):
            off = fx.crd2idx((0, k, m), L).to_py_value()
            line[(off // 8) % 8] = f"lane{m + 16 * k:2d}"
        print("|".join(line))

L = !fly.composed_layout<S<3,3,3> o 0 o (8,8,64):(1,8,64)>
与 e + 8*(c ^ (r%8)) + 64*r 相同: True
lane 0|lane16|lane32|lane48|______|______|______|______
lane17|lane 1|lane49|lane33|______|______|______|______
lane34|lane50|lane 2|lane18|______|______|______|______
lane51|lane35|lane19|lane 3|______|______|______|______
______|______|______|______|lane 4|lane20|lane36|lane52
______|______|______|______|lane21|lane 5|lane53|lane37
______|______|______|______|lane38|lane54|lane 6|lane22
______|______|______|______|lane55|lane39|lane23|lane 7


read 第 0 组是 lane 0-3 和 20-23：swizzle 之后 lane 0-3 在物理 chunk 0-3，lane 20-23 在 4-7，8 个 lane 落在 8 个不同的 bank。其它组同理，用第 4 节的模型算：

In [7]:
for swizzle in [False, True]:
    w = [cycles(write_addrs(p, swizzle), WRITE_GROUPS) for p in range(8)]
    r = [cycles(read_addrs(i, swizzle), READ_GROUPS) for i in range(4)]
    print(f"swizzle={swizzle!s:5}  8 次 ds_write_b128 共 {sum(map(sum, w)):3} 拍，"
          f"4 次 ds_read_b128 共 {sum(map(sum, r)):3} 拍（无冲突各为 64、32）")

swizzle=False  8 次 ds_write_b128 共  64 拍，4 次 ds_read_b128 共 128 拍（无冲突各为 64、32）
swizzle=True   8 次 ds_write_b128 共  64 拍，4 次 ds_read_b128 共  32 拍（无冲突各为 64、32）


### 5.3 在 kernel 里用 swizzle

同一个拷贝 kernel：A（64×64 bf16）先按第 4 节的写法写进 LDS，再按 MFMA 的读法读出来写回 O，结果应和 A 相同。比较四种写法：

| mode | 写法 |
|---|---|
| `none` | 不 swizzle：`plain[None, c, r]` |
| `manual` | 手写 XOR（v1 默认）：`plain[None, c ^ (r % 8), r]` |
| `crd2idx` | 用 swizzle layout 只算地址（v1 的 `"fly"`）：`make_view(add_offset(ptr, crd2idx((0, c, r), L)), 8:1)` |
| `copy` | swizzled tensor + `fx.copy`：`fx.copy(atom, frag, swizzled[None, c, r])` |

为了后面能在另一个进程里用 rocprofv3 跑，`build_copy` 只依赖 `torch`/`fx`/`flyc`/`gpu`。

In [8]:
def build_copy(mode):
    @fx.struct
    class Tile:
        a: fx.Array[fx.BFloat16, 64 * 64, 16]

    @flyc.kernel(known_block_size=(64, 1, 1))
    def lds_copy(A: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        layout = fx.make_ordered_layout((8, 8, 64), 0)     # (chunk 内元素, chunk, 行)
        G = fx.make_view(fx.get_iter(A), layout)
        Ov = fx.make_view(fx.get_iter(O), layout)
        lds = fx.SharedAllocator().allocate(Tile).peek()
        L = fx.make_composed_layout(fx.static(fx.SwizzleType.get(3, 3, 3)), layout)
        plain = lds.a.view(layout)
        swizzled = lds.a.view(L)
        atom = fx.make_copy_atom(fx.UniversalCopy128b(), fx.BFloat16)

        def chunk(c, r):   # 逻辑上第 r 行、第 c 个 chunk 的 16 B
            if fx.const_expr(mode == "manual"):
                return plain[None, c ^ (r % 8), r]
            if fx.const_expr(mode == "crd2idx"):
                return fx.make_view(fx.add_offset(lds.a.ptr, fx.crd2idx((0, c, r), L)), fx.make_layout(8, 1))
            if fx.const_expr(mode == "copy"):
                return swizzled[None, c, r]
            return plain[None, c, r]

        def write(c, r, v):
            if fx.const_expr(mode == "copy"):
                frag = fx.make_fragment_like(plain[None, 0, 0])
                frag.store(v)
                fx.copy(atom, frag, chunk(c, r))
            else:
                chunk(c, r).store(v)

        def read(c, r):
            if fx.const_expr(mode == "copy"):
                frag = fx.make_fragment_like(plain[None, 0, 0])
                fx.copy(atom, chunk(c, r), frag)
                return frag.load()
            return chunk(c, r).load()

        for p in fx.range_constexpr(8):                    # 写：8 个 lane 写一行
            c, r = tid % 8, 8 * p + tid // 8
            write(c, r, G[None, c, r].load())
        gpu.barrier()
        for i in fx.range_constexpr(4):                    # 读：MFMA A 操作数的读法
            for s in fx.range_constexpr(2):
                c, r = tid // 16 + 4 * s, 16 * i + tid % 16
                Ov[None, c, r].store(read(c, r))

    @flyc.jit
    def launch(A: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        lds_copy(A, O).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)

    return launch


A = torch.randn(64, 64, dtype=torch.bfloat16, device="cuda")
O = torch.zeros_like(A)
texts = {}
for mode in ["none", "manual", "crd2idx", "copy"]:
    texts[mode] = isa_both(lambda: build_copy(mode), A, O, stream)
    O.zero_()
    run(build_copy(mode), A, O)
    print(f"{mode:8} GPU 正确: {torch.equal(O, A)}")
    for arch, text in texts[mode].items():
        b = body(text)
        ops = Counter(l.split()[0] for l in b)
        print(f"    {arch}: ds_write_b128 {ops['ds_write_b128']} ds_read_b128 {ops['ds_read_b128']} "
              f"| 带 offset 的 ds {sum('offset:' in l for l in ds(text)):2} "
              f"| v_xor {ops['v_xor_b32_e32'] + ops['v_xor_b32_e64']:2} v_bitop3 {ops['v_bitop3_b32']:2} "
              f"| 指令 {len(b):3} | VGPR {meta(text, 'vgpr_count')}")

none     GPU 正确: True
    gfx942: ds_write_b128 8 ds_read_b128 8 | 带 offset 的 ds 12 | v_xor  0 v_bitop3  0 | 指令  73 | VGPR 38
    gfx950: ds_write_b128 8 ds_read_b128 8 | 带 offset 的 ds 12 | v_xor  0 v_bitop3  1 | 指令  73 | VGPR 38
manual   GPU 正确: True
    gfx942: ds_write_b128 8 ds_read_b128 8 | 带 offset 的 ds 11 | v_xor  3 v_bitop3  0 | 指令  87 | VGPR 44
    gfx950: ds_write_b128 8 ds_read_b128 8 | 带 offset 的 ds 11 | v_xor  0 v_bitop3  3 | 指令  87 | VGPR 44
crd2idx  GPU 正确: True
    gfx942: ds_write_b128 8 ds_read_b128 8 | 带 offset 的 ds  0 | v_xor 16 v_bitop3  0 | 指令 125 | VGPR 56
    gfx950: ds_write_b128 8 ds_read_b128 8 | 带 offset 的 ds  0 | v_xor  0 v_bitop3 16 | 指令 120 | VGPR 56
copy     GPU 正确: True
    gfx942: ds_write_b128 8 ds_read_b128 8 | 带 offset 的 ds  0 | v_xor 16 v_bitop3  0 | 指令 125 | VGPR 56
    gfx950: ds_write_b128 8 ds_read_b128 8 | 带 offset 的 ds  0 | v_xor  0 v_bitop3 16 | 指令 120 | VGPR 56


把 `manual` 和 `copy` 的 `ds_*` 指令并排放在一起：

In [9]:
for arch in ARCHS:
    print(f"==== {arch}")
    side({"manual": ds(texts["manual"][arch]), "copy": ds(texts["copy"][arch])})

==== gfx942
manual                                       | copy
ds_write_b128 v2, v[12:15]                   | ds_write_b128 v14, v[24:27]
ds_write_b128 v2, v[16:19] offset:1024       | ds_write_b128 v17, v[28:31]
ds_write_b128 v2, v[20:23] offset:2048       | ds_write_b128 v7, v[32:35]
ds_write_b128 v2, v[24:27] offset:3072       | ds_write_b128 v8, v[36:39]
ds_write_b128 v2, v[28:31] offset:4096       | ds_write_b128 v9, v[40:43]
ds_write_b128 v2, v[32:35] offset:5120       | ds_write_b128 v10, v[44:47]
ds_write_b128 v2, v[36:39] offset:6144       | ds_write_b128 v11, v[48:51]
ds_write_b128 v2, v[40:43] offset:7168       | ds_write_b128 v18, v[52:55]
ds_read_b128 v[12:15], v11                   | ds_read_b128 v[8:11], v20
ds_read_b128 v[16:19], v11 offset:2048       | ds_read_b128 v[24:27], v21
ds_read_b128 v[20:23], v8                    | ds_read_b128 v[20:23], v22
ds_read_b128 v[24:27], v11 offset:4096       | ds_read_b128 v[28:31], v13
ds_read_b128 v[28:31], v8 offset:2048       

- 四种写法都是 8 条 `ds_write_b128` + 8 条 `ds_read_b128`，结果都正确；swizzle 不改变指令宽度，只改地址。
- `none`：地址差别大多是常数，进了 `offset:`。
- `manual`：XOR 只作用在 chunk 号上。写的 8 次 `r % 8` 相同，XOR 结果不变，8 条写共用一个地址 + `offset:`；读用了 4 个地址寄存器。
- `crd2idx` / `copy`：swizzle 作用在整个 offset 上，编译器没有发现加上行偏移后 XOR 结果不变，16 条 `ds_*` 各自算一个地址，指令和 VGPR 都更多。两者生成的代码规模相同。这和 [v1.py](../kernels/gemm/v1.py) 开头记录的现象一致（大 kernel 里 VGPR +28）。
- gfx942 与 gfx950：`ds_*` 指令和 VGPR 一样；区别是 gfx950 有三输入位运算 `v_bitop3_b32`，所有 XOR 都变成了它，还能顺带合并相邻的 and/or（`none` 里也用了 1 条），`crd2idx`/`copy` 的指令从 125 条降到 120 条。

直接对 swizzled tensor 调 `.load()/.store()` 在 FlyDSL 0.3.4.1 里还不支持：

In [10]:
def load_swizzled(A: fx.Tensor, O: fx.Tensor):
    lds = fx.SharedAllocator().allocate(Tile).peek()
    swizzled = lds.a.view(fx.make_composed_layout(fx.static(fx.SwizzleType.get(3, 3, 3)),
                                                  fx.make_ordered_layout((8, 8, 64), 0)))
    print("swizzled[None, 1, 2]:", swizzled[None, 1, 2].type)
    swizzled[None, 1, 2].load()


try:
    isa(make_launcher(load_swizzled), A, O, stream)
except Exception as e:
    print("error:", re.search(r"MemRefLoadVecOp.*not supported", str(e)).group(0))

error: MemRefLoadVecOp: MemRefType with ComposedLayout is not supported


所以带 swizzle 的 LDS tensor 目前要么用 `fx.copy` + copy atom 读写，要么只用它算地址再建普通视图。另一个思路 `fx.apply_swizzle(ptr, swz)`（把 swizzle 放进指针类型）在这个版本里 `ptr.store` 无法 lower，这里不展开。

## 6. 用硬件计数器验证

用 rocprofv3 读 `SQ_LDS_BANK_CONFLICT`（bank conflict 多花的周期）和 `SQ_INSTS_LDS`（LDS 指令数）。kernel 在另一个进程里跑：把 `build_copy` 的源码写进临时脚本。

注意：计数器只能在本机 gfx950 上测，不是 gfx942。

In [11]:
ROCPROF = shutil.which("rocprofv3") or "/opt/rocm/bin/rocprofv3"


def lds_counters(build, *args, deps=()):
    """在另一个进程里用 rocprofv3 跑 build(*args) 得到的 kernel（名字以 lds_ 开头），返回 LDS 计数器。
    deps：build 里用到的其它函数，源码一起写进脚本"""
    src = "\n".join([
        "import torch",
        "import flydsl.compiler as flyc",
        "import flydsl.expr as fx",
        "from flydsl.expr import gpu",
        *[inspect.getsource(f) for f in deps],
        inspect.getsource(build),
        "A = torch.randn(64, 64, dtype=torch.bfloat16, device='cuda')",
        "O = torch.zeros_like(A)",
        f"{build.__name__}(*{args!r})(A, O, fx.Stream(torch.cuda.current_stream().cuda_stream))",
        "torch.cuda.synchronize()",
        "assert torch.equal(O, A)",
    ])
    with tempfile.TemporaryDirectory() as d:
        script = os.path.join(d, "run.py")
        with open(script, "w") as f:
            f.write(src)
        subprocess.run([ROCPROF, "--pmc", "SQ_INSTS_LDS", "SQ_LDS_BANK_CONFLICT",
                        "--kernel-include-regex", "^lds_", "-d", d, "-o", "pmc", "-f", "csv",
                        "--", sys.executable, script], check=True, capture_output=True, cwd=d)
        path, = glob.glob(f"{d}/**/*counter_collection.csv", recursive=True)
        with open(path) as f:
            return {r["Counter_Name"]: int(float(r["Counter_Value"]))
                    for r in csv.DictReader(f) if r["Kernel_Name"].startswith("lds_")}


for mode in ["none", "manual", "crd2idx", "copy"]:
    print(f"{mode:8}", lds_counters(build_copy, mode))

none     {'SQ_INSTS_LDS': 16, 'SQ_LDS_BANK_CONFLICT': 96}
manual   {'SQ_INSTS_LDS': 16, 'SQ_LDS_BANK_CONFLICT': 0}
crd2idx  {'SQ_INSTS_LDS': 16, 'SQ_LDS_BANK_CONFLICT': 0}
copy     {'SQ_INSTS_LDS': 16, 'SQ_LDS_BANK_CONFLICT': 0}


- 16 条 LDS 指令（8 写 + 8 读）；不 swizzle 时有 bank conflict，三种 swizzle 写法都是 0。
- 本机是 gfx950，bank 结构和 gfx942 不完全相同，冲突周期数不能直接和第 4 节的模型对照，但 "有冲突 / 无冲突" 的结论一致。

## 7. 不经过寄存器：HBM → LDS 的 DMA

前面都是两步：`global_load` 把数据读进 VGPR，再 `ds_write` 写进 LDS。CDNA 还有一类 load 把数据直接写进 LDS（LDS DMA）：

- 指令是 `buffer_load_dword ... lds`（用 buffer descriptor）和 `global_load_lds_dword`（64 位地址）。gfx942 每个 lane 只能搬 1/2/4 B；gfx950 增加了 12/16 B（`_dwordx3` / `_dwordx4`）。
- 数据不进 VGPR，也没有 `ds_write`；完成与否和普通 load 一样用 `vmcnt` 计数。
- LDS 目的地址不由每个 lane 自己给：地址 = `M0`（整个 wave 一个值）+ `lane_id × size`。一条指令让 64 个 lane 按顺序写满一段连续的 `64 × size` 字节（gfx942 是 256 B，gfx950 用 16 B 时是 1 KB）。

FlyDSL 0.3.4.1 里能发出这类指令的接口：

| 方法 | 接口 | 源 | 等待 |
|---|---|---|---|
| `BufferCopyLDS` | `fx.make_copy_atom(fx.rocdl.BufferCopyLDS32b() / BufferCopyLDS128b(), bits)` + `fx.copy` | `fx.rocdl.make_buffer_tensor(...)` | 编译器自动插 `vmcnt` |
| `BufferLoadAsyncLDS` | `fx.rocdl.cdna4.BufferLoadAsyncLDS32b / 96b / 128b()`，用法同上 | buffer tensor | 手动：`fx.rocdl.asyncmark()` + `fx.rocdl.wait_asyncmark(n)` |
| `GlobalLoadAsyncLDS` | `fx.rocdl.cdna4.GlobalLoadAsyncLDS32b / 96b / 128b()` | 普通 global tensor | 手动，同上 |
| `raw_ptr_buffer_load_lds` | `fx.rocdl.raw_ptr_buffer_load_lds(rsrc, lds_ptr, size, voffset, soffset, offset, aux)`；`fx.rocdl.buffer_load_to_lds` 是带默认参数的包装 | `fx.rocdl.get_buffer_rsrc(...)` | 自动 |
| `global_load_lds` | MLIR `rocdl.global_load_lds(gptr, lds_ptr, size, offset)`（`from flydsl._mlir.dialects import rocdl`） | 64 位指针 | 自动 |

另外：

- `rocdl.load_to_lds` / `rocdl.load_async_to_lds` 是通用指针版本，在 gfx942/950 上同样生成 `global_load_lds_dword(x4)`（本实验只确认了指令，没有测它们的等待行为）。
- `tensor_load_to_lds`（TDM）、`cluster_load_async_to_lds`、`rocdl.global_load_async_to_lds_b*` 是 gfx1250 的，gfx942/950 上 LLVM 报 `Cannot select`。
- `cdna4` 命名空间下的两个 async atom，32 位版本在 gfx942 上也能用。
- 在 gfx942 上用 16 B（`BufferCopyLDS128b` 等）时 LLVM 直接 abort（`Do not know how to expand this operator's operand` / `Cannot select`），整个进程退出。所以下面每个架构都用它支持的最大宽度编译：gfx942 4 B，gfx950 16 B。

### 7.1 五种写法对比

1 个 wave 把 A（64×64 bf16 = 8 KB）搬进 LDS，再按行读出来写到 O。`regs` 是对照：同样宽度的 global load + LDS store。第 i 条指令里 lane `t` 搬元素 `[i·64·E + t·E, +E)`，E 是每个 lane 搬的 bf16 个数：LDS 目的地址按 lane 顺序连续，正好符合 DMA 的规则。

In [12]:
from flydsl._mlir.dialects import rocdl as mrocdl

DMA_SIZE = {"gfx942": 4, "gfx950": 16}                # 每个 lane 每条 DMA 最多搬的字节数
LOCAL_SIZE = DMA_SIZE.get(LOCAL_ARCH, 4)                # 本机 GPU 上跑数值用的宽度
METHODS = ["regs", "BufferCopyLDS", "BufferLoadAsyncLDS", "GlobalLoadAsyncLDS",
           "raw_ptr_buffer_load_lds", "global_load_lds"]


def build_fill(method, size, permute_dst=False):
    E = size // 2              # 每个 lane 每条指令搬几个 bf16
    N = 64 * 64 // (64 * E)    # 搬完 8 KB 要几条指令

    @fx.struct
    class Tile:
        a: fx.Array[fx.BFloat16, 64 * 64, 16]

    @flyc.kernel(known_block_size=(64, 1, 1))
    def lds_fill(A: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        lds = fx.SharedAllocator().allocate(Tile).peek()
        if fx.const_expr(method in ("BufferCopyLDS", "BufferLoadAsyncLDS", "raw_ptr_buffer_load_lds")):
            buf = fx.rocdl.make_buffer_tensor(fx.make_view(fx.get_iter(A), fx.make_layout(64 * 64, 1)))
        if fx.const_expr(method == "BufferCopyLDS"):
            atom = fx.make_copy_atom(fx.rocdl.BufferCopyLDS(size * 8), size * 8)
        if fx.const_expr(method == "BufferLoadAsyncLDS"):
            atom = fx.make_copy_atom(fx.rocdl.cdna4.BufferLoadAsyncLDS(size * 8), size * 8)
        if fx.const_expr(method == "GlobalLoadAsyncLDS"):
            atom = fx.make_copy_atom(fx.rocdl.cdna4.GlobalLoadAsyncLDS(size * 8), size * 8)

        for i in fx.range_constexpr(N):
            off = i * 64 * E + tid * E                                # 这个 lane 搬的元素
            lane_dst = (64 - tid) % 64 * E if permute_dst else tid * E   # 7.2 节用：打乱目的地址（lane 0 不变）
            dst = fx.make_view(fx.add_offset(lds.a.ptr, i * 64 * E + lane_dst), fx.make_layout(1, 1))
            src = fx.add_offset(fx.get_iter(A), off)
            if fx.const_expr(method == "regs"):
                fx.make_view(fx.get_iter(dst), fx.make_layout(E, 1)).store(fx.make_view(src, fx.make_layout(E, 1)).load())
            if fx.const_expr(method in ("BufferCopyLDS", "BufferLoadAsyncLDS")):
                fx.copy(atom, fx.make_view(fx.add_offset(fx.get_iter(buf), off), fx.make_layout(1, 1)), dst)
            if fx.const_expr(method == "GlobalLoadAsyncLDS"):
                fx.copy(atom, fx.make_view(src, fx.make_layout(1, 1)), dst)
            if fx.const_expr(method == "raw_ptr_buffer_load_lds"):
                rsrc = fx.rocdl.get_buffer_rsrc(fx.get_iter(buf))
                fx.rocdl.raw_ptr_buffer_load_lds(rsrc, fx.get_iter(dst).llvm_ptr, fx.Int32(size),
                                                 fx.Int32(off * 2), fx.Int32(0), fx.Int32(0), fx.Int32(0))
            if fx.const_expr(method == "global_load_lds"):
                mrocdl.global_load_lds(src.llvm_ptr, fx.get_iter(dst).llvm_ptr, size, 0)
        if fx.const_expr(method in ("BufferLoadAsyncLDS", "GlobalLoadAsyncLDS")):
            fx.rocdl.asyncmark()
            fx.rocdl.wait_asyncmark(0)
        gpu.barrier()
        rows = fx.make_ordered_layout((64, 64), (1, 0))
        Ov = fx.make_view(fx.get_iter(O), rows)
        Ov[tid, None].store(lds.a.view(rows)[tid, None].load())

    @flyc.jit
    def launch(A: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        lds_fill(A, O).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)

    return launch


def move_ops(lines):
    """搬运部分的指令统计：load（带 lds 的加标记）、ds_write、写 M0"""
    ops = Counter()
    for l in lines:
        op = l.split()[0]
        if op.startswith(("global_load", "buffer_load", "ds_write")):
            ops[op + ("(lds)" if l.endswith(" lds") else "")] += 1
        if re.match(r"s_mov\w* m0", l):
            ops["写 m0"] += 1
        if op == "v_readfirstlane_b32":
            ops[op] += 1
    return dict(ops)


def dma_isa(build, *args, io, **kwargs):
    """分别按 gfx942（4 B）/ gfx950（16 B）编译 build(*args, size)，返回 {arch: 指令列表}"""
    return {arch: body(isa(build(*args, DMA_SIZE[arch], **kwargs), *io, arch=arch, quiet=True)) for arch in ARCHS}


def around_first_dma(lines, before=3, after=5):
    first = next(i for i, l in enumerate(lines) if l.endswith(" lds"))
    return lines[first - before:first + after]


A = torch.randn(64, 64, dtype=torch.bfloat16, device="cuda")
O = torch.zeros_like(A)
for method in METHODS:
    O.zero_()
    run(build_fill(method, LOCAL_SIZE), A, O)
    print(f"{method:24} 本机 GPU 正确: {torch.equal(O, A)}")
    for arch, lines in dma_isa(build_fill, method, io=(A, O, stream)).items():
        print(f"    {arch} {DMA_SIZE[arch]:2} B: {move_ops(lines)}")

print("\nBufferCopyLDS 的一段 ISA：")
side({arch: around_first_dma(lines) for arch, lines in dma_isa(build_fill, "BufferCopyLDS", io=(A, O, stream)).items()})

regs                     本机 GPU 正确: True
    gfx942  4 B: {'global_load_dword': 32, 'ds_write2st64_b32': 16}
    gfx950 16 B: {'global_load_dwordx4': 8, 'ds_write_b128': 8}
BufferCopyLDS            本机 GPU 正确: True
    gfx942  4 B: {'v_readfirstlane_b32': 32, '写 m0': 32, 'buffer_load_dword(lds)': 32}
    gfx950 16 B: {'v_readfirstlane_b32': 8, '写 m0': 8, 'buffer_load_dwordx4(lds)': 8}
BufferLoadAsyncLDS       本机 GPU 正确: True
    gfx942  4 B: {'v_readfirstlane_b32': 32, '写 m0': 32, 'buffer_load_dword(lds)': 32}
    gfx950 16 B: {'v_readfirstlane_b32': 8, '写 m0': 8, 'buffer_load_dwordx4(lds)': 8}
GlobalLoadAsyncLDS       本机 GPU 正确: True
    gfx942  4 B: {'v_readfirstlane_b32': 32, '写 m0': 32, 'global_load_lds_dword': 32}
    gfx950 16 B: {'v_readfirstlane_b32': 8, '写 m0': 8, 'global_load_lds_dwordx4': 8}
raw_ptr_buffer_load_lds  本机 GPU 正确: True
    gfx942  4 B: {'v_readfirstlane_b32': 32, '写 m0': 32, 'buffer_load_dword(lds)': 32}
    gfx950 16 B: {'v_readfirstlane_b32': 8, '写 m0': 8, 'buf

- 五种写法结果都正确，没有 `ds_write`。gfx942 是 32 条 DMA（每条 256 B），gfx950 是 8 条（每条 1 KB）。buffer 类（`BufferCopyLDS`、`BufferLoadAsyncLDS`、`raw_ptr_buffer_load_lds`）生成 `buffer_load_dword(x4) ... offen lds`，global 类生成 `global_load_lds_dword(x4)`。
- `regs` 在 gfx942 上是 32 条 `global_load_dword` + 16 条 `ds_write2st64_b32`（两条相距 64×4 B 的写合并成一条），在 gfx950 上是 8 条 `global_load_dwordx4` + 8 条 `ds_write_b128`。数据都要先占用 VGPR，16 B 的 load 每条在飞时占 4 个 VGPR。
- DMA 每条指令前都要写一次 `m0`（LDS 基址）。这里每个 lane 传的是自己的 LDS 地址，编译器先用 `v_readfirstlane_b32` 取 lane 0 的值再 `s_mov_b32 m0`；如果传的是编译期常数基址，就只是一条 `s_movk_i32 m0, 常数`。

### 7.2 LDS 目的地址只看 lane 0

上面每个 lane 传的 `dst` 是自己的地址 `base + t·E`，结果正确只是因为它和硬件的规则一致。把目的地址打乱（lane t 写到 `base + ((64 - t) % 64)·E`，lane 0 不变）再试：

In [13]:
N = 64 * 64 // (64 * (LOCAL_SIZE // 2))
t = torch.arange(64)
permuted = torch.empty_like(A).view(N, 64, -1)
permuted[:, (64 - t) % 64] = A.view(N, 64, -1)          # 按每个 lane 自己的地址写应得到的结果
for method in ["regs", "BufferCopyLDS"]:
    O.zero_()
    run(build_fill(method, LOCAL_SIZE, permute_dst=True), A, O)
    print(f"{method:14} 等于打乱后的 A: {torch.equal(O, permuted.view(64, 64))!s:5} | 等于 A: {torch.equal(O, A)}")

side({arch: around_first_dma(lines, 3, 3)
      for arch, lines in dma_isa(build_fill, "BufferCopyLDS", io=(A, O, stream), permute_dst=True).items()})

regs           等于打乱后的 A: True  | 等于 A: False
BufferCopyLDS  等于打乱后的 A: False | 等于 A: True
gfx942                                       | gfx950
s_mov_b32 m0, s2                             | s_mov_b32 m0, s2
v_readfirstlane_b32 s2, v3                   | v_readfirstlane_b32 s2, v3
v_or_b32_e32 v3, 0x200, v2                   | v_or_b32_e32 v3, 0x800, v2
buffer_load_dword v1, s[4:7], 0 offen lds    | buffer_load_dwordx4 v1, s[4:7], 0 offen lds
v_or_b32_e32 v4, 0x100, v1                   | v_or_b32_e32 v4, 0x400, v1
s_mov_b32 m0, s2                             | s_mov_b32 m0, s2


- `regs`：每个 lane 按自己的地址写，LDS 里是打乱后的 A。
- `BufferCopyLDS`：编译器用 `v_readfirstlane_b32` 取 lane 0 的地址写进 `m0`，其余 lane 的地址被忽略，硬件仍按 lane 顺序写，结果还是原样的 A，没有任何提示。如果 lane 0 的地址也不是基址，整段数据还会被写偏。

所以用 DMA 时，想要什么 LDS 布局只能通过**选择源地址**来实现：硬件决定每个 lane 写到哪里，我们决定它从哪里读。

### 7.3 同步与异步：编译器插入的等待

4 个 wave 先后发出两组 DMA（第 0 组填 LDS 前半，第 1 组填后半），然后先读前半、再读后半。理想情况下读前半时只需等第 0 组，第 1 组可以继续在飞——这就是多级流水的基础。

- `sync`：`BufferCopyLDS`，不写任何等待；
- `async`：`BufferLoadAsyncLDS`，每组后 `asyncmark()`，读前半前 `wait_asyncmark(1)`（最多剩 1 组未完成），读后半前 `wait_asyncmark(0)`。

每个 lane 读的是另一个 wave 搬的数据，所以必须等 DMA 完成再 `barrier`。

In [14]:
def build_two_stage(method, size):
    E = size // 2
    HALF = 4096                       # 每半 4096 个 bf16（8 KB）
    K = HALF // (4 * 64 * E)          # 每个 wave 每组发几条 DMA
    J = HALF // 8 // 256              # 读回时每个 lane 读几个 16 B

    @fx.struct
    class Smem:
        buf: fx.Array[fx.BFloat16, 2 * HALF, 16]

    @flyc.kernel(known_block_size=(256, 1, 1))
    def lds_two_stage(A: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        wave, lane = tid // 64, tid % 64
        lds = fx.SharedAllocator().allocate(Smem).peek()
        buf = fx.rocdl.make_buffer_tensor(fx.make_view(fx.get_iter(A), fx.make_layout(2 * HALF, 1)))
        atom = fx.make_copy_atom(fx.rocdl.BufferCopyLDS(size * 8) if method == "sync"
                                 else fx.rocdl.cdna4.BufferLoadAsyncLDS(size * 8), size * 8)

        def issue(h):                  # 第 h 组：填 LDS 的第 h 半
            for k in fx.range_constexpr(K):
                base = h * HALF + (k * 4 + wave) * 64 * E
                dst = fx.make_view(fx.add_offset(lds.buf.ptr, base + lane * E), fx.make_layout(1, 1))
                src = fx.make_view(fx.add_offset(fx.get_iter(buf), base + lane * E), fx.make_layout(1, 1))
                fx.copy(atom, src, dst)

        layout = fx.make_ordered_layout((8, J, 256, 2), 0)
        T = lds.buf.view(layout)
        Ov = fx.make_view(fx.get_iter(O), layout)
        other = (tid + 64) % 256       # 读下一个 wave 搬的数据

        issue(0)
        if fx.const_expr(method == "async"):
            fx.rocdl.asyncmark()
        issue(1)
        if fx.const_expr(method == "async"):
            fx.rocdl.asyncmark()
            fx.rocdl.wait_asyncmark(1)
        gpu.barrier()
        for j in fx.range_constexpr(J):
            Ov[None, j, tid, 0].store(T[None, j, other, 0].load())
        if fx.const_expr(method == "async"):
            fx.rocdl.wait_asyncmark(0)
        gpu.barrier()
        for j in fx.range_constexpr(J):
            Ov[None, j, tid, 1].store(T[None, j, other, 1].load())

    @flyc.jit
    def launch(A: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        lds_two_stage(A, O).launch(grid=(1, 1, 1), block=(256, 1, 1), stream=stream)

    return launch


def squeeze(lines):
    """只留 DMA / 等待 / barrier，连续的 DMA 合并成一行"""
    out = []
    for l in lines:
        if l.endswith(" lds"):
            if out and out[-1][0] == "dma":
                out[-1][1] += 1
            else:
                out.append(["dma", 1])
        elif re.match(r"s_waitcnt.*vmcnt|s_barrier", l):
            out.append([l, 0])
    return [f"buffer_load ... lds × {n}" if l == "dma" else l for l, n in out]


A2 = torch.randn(2 * 4096, dtype=torch.bfloat16, device="cuda")
O2 = torch.zeros_like(A2)
expect = A2.view(2, 256, -1).roll(-64, dims=1)
for method in ["sync", "async"]:
    O2.zero_()
    run(build_two_stage(method, LOCAL_SIZE), A2, O2)
    print(f"==== {method}: 本机 GPU 正确 = {torch.equal(O2.view(2, 256, -1), expect)}")
    side({arch: squeeze(lines) for arch, lines in dma_isa(build_two_stage, method, io=(A2, O2, stream)).items()},
         width=32)

==== sync: 本机 GPU 正确 = True
gfx942                           | gfx950
buffer_load ... lds × 16         | buffer_load ... lds × 4
s_waitcnt vmcnt(0) lgkmcnt(0)    | s_waitcnt vmcnt(0) lgkmcnt(0)
s_barrier                        | s_barrier
s_barrier                        | s_barrier
==== async: 本机 GPU 正确 = True
gfx942                           | gfx950
buffer_load ... lds × 16         | buffer_load ... lds × 4
s_waitcnt vmcnt(8) lgkmcnt(0)    | s_waitcnt vmcnt(2) lgkmcnt(0)
s_barrier                        | s_barrier
s_waitcnt vmcnt(2)               | s_waitcnt vmcnt(2)
s_barrier                        | s_barrier


- 每个 wave 每组的 DMA 条数：gfx942（4 B）是 8 条，两组共 16 条；gfx950（16 B）是 2 条，两组共 4 条。
- `sync`：两种架构都在第一个 barrier 前插了 `vmcnt(0)`，把两组都等完。编译器不知道接下来只读前半，只能保守地全等。
- `async`：`wait_asyncmark(1)` 变成只等第 0 组的计数——gfx942 `vmcnt(8)`、gfx950 `vmcnt(2)`，正好是第 1 组的条数，这些 DMA 可以继续在飞。`wait_asyncmark(0)` 两边都变成 `vmcnt(2)`：中间又发了 2 条 `global_store`，gfx9 的 `vmcnt` 把 store 也计在内，编译器把组数换算成了正确的计数。
- 换句话说：同步版本写起来最省事，但无法和下一个 tile 的搬运重叠；要做 double buffer，用 async 版本自己分组、自己等。

### 7.4 DMA + swizzle

DMA 写 LDS 的顺序是固定的，所以第 5 节的 swizzle 要反过来做：lane 写到物理位置（第 `r` 行的第 `p` 个 chunk），就让它从全局内存里读逻辑 chunk `p ^ (r % 8)`。XOR 是自己的逆，读 LDS 时仍用第 5 节的 `c ^ (r % 8)`。这也是 pyhip `test_async_copy.py` 的做法：swizzle 加在 buffer load 的源地址和 LDS 读地址上。

In [15]:
def build_dma_swizzle(swizzle, size):
    E = size // 2
    N = 64 * 64 // (64 * E)

    @fx.struct
    class Tile:
        a: fx.Array[fx.BFloat16, 64 * 64, 16]

    @flyc.kernel(known_block_size=(64, 1, 1))
    def lds_dma(A: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        lds = fx.SharedAllocator().allocate(Tile).peek()
        buf = fx.rocdl.make_buffer_tensor(fx.make_view(fx.get_iter(A), fx.make_layout(64 * 64, 1)))
        atom = fx.make_copy_atom(fx.rocdl.BufferCopyLDS(size * 8), size * 8)
        for i in fx.range_constexpr(N):
            q = i * 64 * E + tid * E                  # 硬件决定：这个 lane 写到 LDS 的元素位置
            r, col = q // 64, q % 64
            src = r * 64 + ((col // 8) ^ (r % 8)) * 8 + col % 8 if swizzle else q   # 反过来选源地址
            fx.copy(atom, fx.make_view(fx.add_offset(fx.get_iter(buf), src), fx.make_layout(1, 1)),
                    fx.make_view(fx.add_offset(lds.a.ptr, q), fx.make_layout(1, 1)))
        gpu.barrier()
        layout = fx.make_ordered_layout((8, 8, 64), 0)
        plain = lds.a.view(layout)
        Ov = fx.make_view(fx.get_iter(O), layout)
        for i in fx.range_constexpr(4):               # 读：和第 5 节一样的 MFMA 读法
            for s in fx.range_constexpr(2):
                c, r = tid // 16 + 4 * s, 16 * i + tid % 16
                Ov[None, c, r].store(plain[None, c ^ (r % 8) if swizzle else c, r].load())

    @flyc.jit
    def launch(A: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        lds_dma(A, O).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)

    return launch


for swizzle in [False, True]:
    O.zero_()
    run(build_dma_swizzle(swizzle, LOCAL_SIZE), A, O)
    print(f"swizzle={swizzle!s:5} 本机 GPU 正确: {torch.equal(O, A)} | 本机计数器: "
          f"{lds_counters(build_dma_swizzle, swizzle, LOCAL_SIZE)}")
    for arch, lines in dma_isa(build_dma_swizzle, swizzle, io=(A, O, stream)).items():
        print(f"    {arch} {DMA_SIZE[arch]:2} B: {move_ops(lines)}，共 {len(lines)} 条指令")

swizzle=False 本机 GPU 正确: True | 本机计数器: {'SQ_INSTS_LDS': 8, 'SQ_LDS_BANK_CONFLICT': 96}
    gfx942  4 B: {'v_readfirstlane_b32': 32, '写 m0': 32, 'buffer_load_dword(lds)': 32}，共 226 条指令
    gfx950 16 B: {'v_readfirstlane_b32': 8, '写 m0': 8, 'buffer_load_dwordx4(lds)': 8}，共 85 条指令
swizzle=True  本机 GPU 正确: True | 本机计数器: {'SQ_INSTS_LDS': 8, 'SQ_LDS_BANK_CONFLICT': 0}
    gfx942  4 B: {'v_readfirstlane_b32': 32, '写 m0': 32, 'buffer_load_dword(lds)': 32}，共 418 条指令
    gfx950 16 B: {'v_readfirstlane_b32': 8, '写 m0': 8, 'buffer_load_dwordx4(lds)': 8}，共 124 条指令


- 两种写法的 DMA 指令一样，swizzle 只改了每个 lane 的源地址和 LDS 读地址。每条 DMA 都要单独算一次带 XOR 的源地址：gfx942 每条只搬 4 B，32 条 DMA 让总指令数从 226 增加到 418；gfx950 只有 8 条，从 85 增加到 124。
- 不 swizzle 时 MFMA 读法仍有 bank conflict，swizzle 后为 0。`SQ_INSTS_LDS` 只数了 8 条 `ds_read`，DMA 不算 LDS 指令。
- swizzle 后每 8 个 lane 读的仍是同一行的 128 B，只是顺序打乱了。

### 7.5 封装：`dma_to_lds(src, lds, bits)`

7.2 和 7.4 暴露了 `fx.copy` + DMA atom 的两个问题：

- 接口看起来是每个 lane 给自己的目的地址，实际只用 lane 0 的，写错了也不报错；
- swizzle 要自己把 LDS 的物理位置换算成源地址，每条 `fx.copy` 都要写两个 `make_view(add_offset(...))`。

把硬件规则直接写进接口：

- LDS 侧传一个**普通、紧凑的 tensor**，它的 offset 顺序就是写入顺序：第 i 条指令写 offset `[i·64·E, (i+1)·64·E)`，lane t 写其中第 t 段。封装里只算 wave 级的基址，没有 per-lane 目的地址。
- 源侧传一个和 LDS 逻辑 shape 相同的 global tensor。lane 写 LDS 的 offset `q`，就读 `src[idx2crd(q, lds.layout)]`。想要 swizzle，就把 swizzle 加在 src 的 layout 上（composed layout），和第 5 节用同一个 `Swizzle(3, 3, 3)`。

这是一个普通 Python 函数，在 kernel trace 时调用：

In [16]:
def dma_to_lds(src, lds, bits, *, sync=True, num_waves=1):
    """HBM -> LDS DMA：把 src 搬进 lds（两者逻辑 shape 相同）。

    写入次序由硬件决定：第 i 条指令由一个 wave 写 lds 的 offset [i*64*E, (i+1)*64*E)，
    lane t 写其中第 t 段 E 个元素（E = bits / 元素位宽）。
    - lds：LDS 上的普通紧凑 tensor，不能带 swizzle；
    - src：global tensor，可以是 composed layout（如 swizzle）。lane 写 lds 的 offset q，
      数据取自 src[idx2crd(q, lds.layout)]；每个 lane 的 E 个元素在 src 里必须连续；
    - num_waves > 1：各 wave 轮流负责相邻的段，基址用 readfirstlane 放进 SGPR；
    - sync=False：用 async atom，调用方自己 asyncmark / wait_asyncmark。
    """
    E = bits // src.element_type.width
    total = fx.size(lds.layout).to_py_value()
    assert fx.cosize(lds.layout).to_py_value() == total, "lds 必须是紧凑 layout：DMA 按物理顺序连续写"
    assert total % (64 * E * num_waves) == 0
    atom = fx.make_copy_atom(fx.rocdl.BufferCopyLDS(bits) if sync else fx.rocdl.cdna4.BufferLoadAsyncLDS(bits), bits)
    buf = fx.rocdl.make_buffer_ptr(fx.get_iter(src))
    lane = fx.thread_idx.x % 64
    wave = fx.thread_idx.x // 64 if num_waves > 1 else 0
    for k in range(total // (64 * E * num_waves)):
        base = (k * num_waves + wave) * 64 * E          # 这条指令写的 LDS 起点，wave 内相同
        if num_waves > 1:
            base = fx.rocdl.readfirstlane(fx.Int32.ir_type, base)
        off = fx.crd2idx(fx.idx2crd(base + lane * E, lds.layout), src.layout)   # 本 lane 的源地址
        fx.copy(atom, fx.make_view(fx.add_offset(buf, off), fx.make_layout(1, 1)),
                fx.make_view(fx.add_offset(fx.get_iter(lds), base), fx.make_layout(1, 1)))

用它重写 7.4 的 kernel，并试一下 4 个 wave 的版本。搬运部分只有两行：

```python
src = fx.make_view(fx.get_iter(A), fx.make_composed_layout(swz, layout))   # swizzle 加在源 tensor 上
dma_to_lds(src, lds.a.view(layout), size * 8, num_waves=num_waves)       # LDS 侧是普通 layout
```

In [17]:
def build_dma_wrapped(swizzle, size, num_waves=1):
    @fx.struct
    class Tile:
        a: fx.Array[fx.BFloat16, 64 * 64, 16]

    @flyc.kernel(known_block_size=(64 * num_waves, 1, 1))
    def lds_dma_wrapped(A: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        lane, wave = tid % 64, tid // 64
        layout = fx.make_ordered_layout((8, 8, 64), 0)     # (chunk 内元素, chunk, 行)
        swz = fx.static(fx.SwizzleType.get(3, 3, 3))
        lds = fx.SharedAllocator().allocate(Tile).peek()
        src = fx.make_view(fx.get_iter(A), fx.make_composed_layout(swz, layout) if swizzle else layout)
        dma_to_lds(src, lds.a.view(layout), size * 8, num_waves=num_waves)
        gpu.barrier()
        plain = lds.a.view(layout)
        Ov = fx.make_view(fx.get_iter(O), layout)
        for i in fx.range_constexpr(4 // num_waves):       # 读：和第 5 节一样的 MFMA 读法，各 wave 分行
            for s in fx.range_constexpr(2):
                c, r = lane // 16 + 4 * s, 16 * (i * num_waves + wave) + lane % 16
                Ov[None, c, r].store(plain[None, c ^ (r % 8) if swizzle else c, r].load())

    @flyc.jit
    def launch(A: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        lds_dma_wrapped(A, O).launch(grid=(1, 1, 1), block=(64 * num_waves, 1, 1), stream=stream)

    return launch


for swizzle in [False, True]:
    for num_waves in [1, 4]:
        O.zero_()
        run(build_dma_wrapped(swizzle, LOCAL_SIZE, num_waves), A, O)
        print(f"swizzle={swizzle!s:5} num_waves={num_waves} 本机 GPU 正确: {torch.equal(O, A)}")
    counters = lds_counters(build_dma_wrapped, swizzle, LOCAL_SIZE, deps=[dma_to_lds])
    print(f"    本机计数器（1 wave）: {counters}")
    old = dma_isa(build_dma_swizzle, swizzle, io=(A, O, stream))
    new = dma_isa(build_dma_wrapped, swizzle, io=(A, O, stream))
    for arch in ARCHS:
        print(f"    {arch} {DMA_SIZE[arch]:2} B: 7.4 写法 {len(old[arch])} 条指令 -> dma_to_lds {len(new[arch])} 条，"
              f"{move_ops(new[arch])}")

print("\nswizzle=True，1 个 wave 的 DMA 部分：")
side({arch: around_first_dma(lines, 4, 6) for arch, lines in dma_isa(build_dma_wrapped, True, io=(A, O, stream)).items()})
print("\n4 个 wave 时 v_readfirstlane 条数：",
      {arch: sum(l.startswith("v_readfirstlane") for l in lines)
       for arch, lines in dma_isa(build_dma_wrapped, True, io=(A, O, stream), num_waves=4).items()})

swizzle=False num_waves=1 本机 GPU 正确: True
swizzle=False num_waves=4 本机 GPU 正确: True
    本机计数器（1 wave）: {'SQ_INSTS_LDS': 8, 'SQ_LDS_BANK_CONFLICT': 96}
    gfx942  4 B: 7.4 写法 226 条指令 -> dma_to_lds 166 条，{'写 m0': 32, 'buffer_load_dword(lds)': 32}
    gfx950 16 B: 7.4 写法 85 条指令 -> dma_to_lds 71 条，{'写 m0': 8, 'buffer_load_dwordx4(lds)': 8}
swizzle=True  num_waves=1 本机 GPU 正确: True
swizzle=True  num_waves=4 本机 GPU 正确: True
    本机计数器（1 wave）: {'SQ_INSTS_LDS': 8, 'SQ_LDS_BANK_CONFLICT': 0}
    gfx942  4 B: 7.4 写法 418 条指令 -> dma_to_lds 255 条，{'写 m0': 32, 'buffer_load_dword(lds)': 32}
    gfx950 16 B: 7.4 写法 124 条指令 -> dma_to_lds 100 条，{'写 m0': 8, 'buffer_load_dwordx4(lds)': 8}

swizzle=True，1 个 wave 的 DMA 部分：
gfx942                                       | gfx950
s_and_b32 s5, s5, 0xffff                     | v_and_b32_e32 v2, 56, v0
v_lshlrev_b32_e32 v5, 1, v5                  | v_lshlrev_b32_e32 v3, 1, v3
s_mov_b32 m0, 0                              | s_mov_b32 m0, 0
v_and_b32_e32 v2, 62, v1

- 结果都正确，4 个 wave 分工也正确；swizzle 后 bank conflict 仍为 0。
- 指令比 7.4 少：LDS 基址是编译期常数，`m0` 用 `s_movk_i32` 直接设，不再每条 DMA 前 `v_readfirstlane`；源地址的 swizzle 由 `crd2idx` 在 composed layout 上算出来，常数部分被折叠。
- 4 个 wave 时每个 wave 的基址是运行时值，只在开头 `readfirstlane` 一次。
- 误用会报错而不是静默出错：把带 swizzle 的 LDS view 传给 `lds`，`idx2crd` 会报 `expected LayoutType for layout, got '!fly.composed_layout<...>'`；带 padding 的 LDS layout 会触发 cosize 的 assert。
- 前提：每个 lane 的 E 个元素在 src 里连续。`Swizzle(3, 3, 3)` 保留低 3 位（8 个 bf16），所以 gfx950 的 16 B、gfx942 的 4 B 都满足。封装没有检查这一点。
- src 的 swizzle 要和 LDS 读取时用的 swizzle 对应：这里 tile 的行长等于 global 的行长，两边都是 `Swizzle(3, 3, 3)`。如果 global 的行更长，应该先把 global tile 切成和 LDS 同样的 layout（例如用 `make_view` 把 tile 视为 `(8, 8, 64)` 加上行 stride），再套 swizzle。

## 总结

| 问题 | 结论 |
|---|---|
| 定义数据结构 | `@fx.struct` / `@fx.union` + 类型注解，`fx.Array[T, N, align]`、`fx.Align[T, n]`；size/align/offset 规则同 C |
| 分配到 LDS | `st = fx.SharedAllocator(static=...).allocate(Smem)` 得到 `Storage`；标量 `st.x.poke/peek`，数组 `st.a.peek()` 得到指针，`.view(layout)` 得到 `!fly.memref<..., shared, ...>` |
| `static=False` | 一块 dynamic LDS，offset 与 C++ 相同，常量 offset 进 `ds_*` 的 `offset:`；metadata 里 LDS 大小为 0，launch 时传入 |
| `static=True`（默认） | 每个叶子字段一个 LDS 变量，地址由 LLVM 决定，不保留字段顺序；`group_segment_fixed_size` 是实际用量 |
| LDS `.load()/.store()` | 连续 8/4/2/1 个 bf16 → `ds_read_b128` / `b64` / `b32` / `u16`（写同理）；不连续拆成逐元素；地址 = VGPR + 16 位立即数 |
| bank conflict | gfx942：32 个 4 B bank；b128 访问每 8 个 lane 一拍。行距 128 B 时 MFMA 读法（16 行同一 chunk）是 4-way conflict（模型）；gfx950 实测同样有冲突 |
| swizzle | `fx.SwizzleType.get(3, 3, 3)` + `fx.make_composed_layout` 等价于 `chunk ^ (row % 8)`，读写都无冲突（gfx942 模型、gfx950 计数器） |
| 用法与 ISA | swizzled tensor 不能直接 `.load()/.store()`；可用 `fx.copy` 或 `crd2idx` 算地址，指令仍是 `ds_*_b128`，但地址不能共用，VGPR 更多；手写 XOR 只作用在 chunk 号上，生成的地址计算最少。gfx950 用 `v_bitop3_b32` 合并 XOR，指令略少 |
| HBM → LDS DMA | `BufferCopyLDS` / `BufferLoadAsyncLDS` / `GlobalLoadAsyncLDS` copy atom，或 `raw_ptr_buffer_load_lds` / `rocdl.global_load_lds`；数据不进 VGPR、没有 `ds_write`。gfx942 每 lane 最多 4 B（`buffer_load_dword ... lds` / `global_load_lds_dword`），gfx950 可到 16 B（`_dwordx4`） |
| DMA 的 LDS 地址 | `m0`（取 lane 0）+ `lane × size`，其余 lane 给的地址被忽略且不报错；LDS 布局（包括 swizzle）只能通过选择源地址实现 |
| 同步 vs 异步 | `BufferCopyLDS` 由编译器插 `vmcnt(0)`，把所有 DMA 都等完；`*AsyncLDS` + `asyncmark` / `wait_asyncmark(n)` 只等指定的组（例子里 gfx942 `vmcnt(8)`、gfx950 `vmcnt(2)`），可以做多级流水 |
| DMA 封装 | `dma_to_lds(src, lds, bits)`：LDS 侧只接受紧凑的普通 tensor，按 wave 级基址依次写；lane 写 offset q 就读 `src[idx2crd(q, lds.layout)]`，swizzle 加在 src 的 layout 上。代码更短、指令更少，误用会报错 |